In [0]:
from pyspark.sql import functions as F

spark.sql("USE plworkforce_catalog.002_silver")

# Read Bronze GL
df = spark.table("plworkforce_catalog.001_bronze.general_ledgers")

# Read account types from Bronze Accounts
accounts = spark.table("plworkforce_catalog.001_bronze.accounts").select(
    "account_id", "account_type"
)

# Correct Silver Transformation
df_clean = (
    df
    .withColumnRenamed("gl_id", "transaction_id")
    
    # Parse timestamps
    .withColumn("entry_date", F.to_timestamp("entry_date", "dd-MM-yyyy HH:mm"))
    .withColumn("posting_date", F.to_timestamp("posting_date", "dd-MM-yyyy HH:mm"))
    
    # Cast numeric fields
    .withColumn("debit_amount", F.col("debit_amount").cast("double"))
    .withColumn("credit_amount", F.col("credit_amount").cast("double"))
    
    # JOIN to get the account_type
    .join(accounts, "account_id", "left")
    
    # Correct accounting logic
    .withColumn(
        "amount",
        F.when(F.col("account_type") == "Revenue", F.col("credit_amount"))
         .when(F.col("account_type") == "Cogs", F.col("debit_amount"))
         .when(F.col("account_type") == "Expense", F.col("debit_amount"))
         .otherwise(F.col("debit_amount") - F.col("credit_amount"))
    )
    
    # Date parts
    .withColumn("year", F.year("posting_date"))
    .withColumn("month", F.month("posting_date"))
    .withColumn("year_month", F.date_format("posting_date", "yyyy-MM"))
    
    # Deduplicate
    .dropDuplicates(["transaction_id"])
)

# Save to Silver
df_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("general_ledgers")

print("Silver GL updated with correct accounting logic.")